# 03 · Evaluation Query Set & Ground Truth Definition
**This is the most important General notebook.**

It defines the 50 evaluation queries and their ground-truth relevant paper IDs
ONCE, independently of any retrieval system. All scale experiments load this file.

This directly fixes the circular pseudo-relevance problem from the old notebooks.


In [ ]:

import os, json, pandas as pd

GROUND_TRUTH_PATH = "../../1_data/eval/ground_truth.json"
os.makedirs("../../1_data/eval", exist_ok=True)


In [ ]:

# The 50 evaluation queries (same as Tier 2 set, stratified)
QUERIES = [
    # Imaging / Visual (10)
    "What CT imaging findings are characteristic of COVID-19 pneumonia?",
    "How does chest X-ray appearance differ between COVID-19 and bacterial pneumonia?",
    "What ultrasound features are observed in COVID-19 patients?",
    "How does CT severity score correlate with COVID-19 outcomes?",
    "What are the radiological features of COVID-19 in children?",
    "How does ground-glass opacity evolve over the course of COVID-19?",
    "What MRI findings have been reported in COVID-19 neurological complications?",
    "How is CT used to monitor COVID-19 treatment response?",
    "What are the pulmonary sequelae visible on CT after COVID-19 recovery?",
    "What deep learning approaches have been applied to COVID-19 CT diagnosis?",
    # Molecular / Mechanistic (10)
    "How does SARS-CoV-2 spike protein bind to ACE2 receptors?",
    "What is the role of the furin cleavage site in SARS-CoV-2 infection?",
    "How does SARS-CoV-2 evade the innate immune response?",
    "What mutations in the spike protein affect vaccine efficacy?",
    "How does TMPRSS2 facilitate SARS-CoV-2 cell entry?",
    "What is the mechanism of SARS-CoV-2 RNA replication?",
    "How do neutralising antibodies target the SARS-CoV-2 receptor binding domain?",
    "What is the role of the nucleocapsid protein in SARS-CoV-2?",
    "How does SARS-CoV-2 affect the renin-angiotensin system?",
    "What structural features distinguish SARS-CoV-2 from SARS-CoV-1?",
    # Clinical Outcomes (10)
    "What are the risk factors for severe COVID-19 disease?",
    "What role does IL-6 play in COVID-19 cytokine storm?",
    "What are the cardiac complications associated with COVID-19?",
    "How is long COVID defined and what are its most common symptoms?",
    "What are the neurological complications of COVID-19?",
    "How does COVID-19 affect patients with diabetes?",
    "What is the mortality rate of COVID-19 in ICU patients?",
    "How does age affect COVID-19 severity and outcomes?",
    "What are the renal complications of COVID-19?",
    "What coagulation abnormalities are observed in severe COVID-19?",
    # Treatment / Intervention (10)
    "How effective are mRNA vaccines against COVID-19 variants?",
    "What antiviral drugs have shown efficacy against SARS-CoV-2?",
    "What is the evidence for dexamethasone in severe COVID-19?",
    "How effective is remdesivir for COVID-19 treatment?",
    "What is the role of convalescent plasma in COVID-19 treatment?",
    "How do monoclonal antibodies work against SARS-CoV-2?",
    "What ventilation strategies are used for COVID-19 ARDS?",
    "What is the evidence for prone positioning in COVID-19?",
    "How effective are COVID-19 vaccines in immunocompromised patients?",
    "What is the evidence for anticoagulation therapy in COVID-19?",
    # Dataset / Methodology (5)
    "How was the CORD-19 dataset constructed and what does it contain?",
    "What NLP methods have been applied to COVID-19 literature?",
    "How have information retrieval systems been used to search COVID-19 research?",
    "What are the main limitations of COVID-19 observational studies?",
    "How have preprints affected COVID-19 research dissemination?",
    # Cross-domain Synthesis (5)
    "How do socioeconomic factors affect COVID-19 outcomes?",
    "What is the relationship between air pollution and COVID-19 severity?",
    "How has COVID-19 affected mental health at a population level?",
    "What lessons from previous coronavirus outbreaks apply to COVID-19?",
    "How has COVID-19 impacted healthcare delivery and access?",
]

print(f"Total queries: {len(QUERIES)}")
assert len(QUERIES) == 50, "Must have exactly 50 queries"


In [ ]:

# ─────────────────────────────────────────────────────────────────────────
# GROUND TRUTH DEFINITION
#
# TWO-PHASE APPROACH:
#
# Phase 1 (immediate, interim): Use the scale_50K hybrid top-3 results as
#   pseudo-ground-truth. This is less circular than old approach because:
#   - Ground truth defined at LARGEST scale (50K), applied to ALL smaller scales
#   - Smaller scales are evaluated against a FIXED external standard
#   - Still pseudo-relevance but no longer within-scale circular
#
# Phase 2 (before submission): Replace with manual annotations.
#   annotation CSV: 4_results/tier2/passages_for_annotation.csv
#   Target: 50 queries × 3 passages = 150 binary relevance judgments
#   Time: ~2-3 hours
#
# This notebook first tries to load manual annotations if available,
# and falls back to the Phase 1 pseudo-labels if not.
# ─────────────────────────────────────────────────────────────────────────

MANUAL_ANNOTATION_PATH = "../../4_results/tier2/passages_for_annotation.csv"

if os.path.exists(MANUAL_ANNOTATION_PATH):
    ann = pd.read_csv(MANUAL_ANNOTATION_PATH)
    print(f"Annotation file found: {len(ann)} rows")
    print("Columns:", ann.columns.tolist())
    # TODO: After manual annotation, add 'relevant' column (1/0) and load here
    # For now, check if annotation is complete
    if 'relevant' in ann.columns:
        print("Manual annotations present — using as ground truth")
        gt_source = "manual"
    else:
        print("No 'relevant' column yet — manual annotation needed")
        print("→ Using Phase 1 interim ground truth (scale_50K pseudo-labels)")
        gt_source = "pseudo_50K"
else:
    print("Annotation file not found — using Phase 1 interim ground truth")
    gt_source = "pseudo_50K"

print(f"\nGround truth source: {gt_source}")


In [ ]:

# Load the Phase 1 interim ground truth from scale_50K results
# (These are the old Tier 2 results — already the best we have)
import pandas as pd

TIER2_RECALL_CSV = "../../4_results/tier2/recall_at_k_tier2.csv"

if os.path.exists(TIER2_RECALL_CSV):
    df_t2 = pd.read_csv(TIER2_RECALL_CSV)
    print(f"Loaded scale_50K results: {len(df_t2)} queries")
    print(df_t2.head(3))
else:
    print("scale_50K results not yet available.")
    print("Run scale_50K notebooks first, then return here to build ground truth.")


In [ ]:

# ─── MANUAL ANNOTATION HELPER ────────────────────────────────────────────────
# When you are ready to do manual annotation:
# 1. Open 4_results/tier2/passages_for_annotation.csv
# 2. For each row, add a 'relevant' column: 1 if relevant, 0 if not
# 3. Save back to the same file
# 4. Re-run this cell to build the manual ground truth
# ─────────────────────────────────────────────────────────────────────────────

# After annotation is complete, run this to save the final ground truth:
# ground_truth = {}
# for query, group in annotated_df.groupby('query'):
#     relevant_ids = group[group['relevant'] == 1]['cord_uid'].tolist()
#     ground_truth[query] = relevant_ids
# with open(GROUND_TRUTH_PATH, "w") as f:
#     json.dump(ground_truth, f, indent=2)
# print(f"Ground truth saved: {len(ground_truth)} queries")

print("Ground truth notebook ready.")
print("Next step: manually annotate passages_for_annotation.csv,")
print("then re-run this notebook to save ground_truth.json")
